# Paper 4 — Run all after connecting a GPU
1. Select **Runtime → Change runtime type → GPU**.
2. Select **Runtime → Run all**, and authorize Google Drive when prompted.
3. Leave the scientific settings unchanged. After a normal interruption, reconnect a GPU and run all again to resume the same configuration.

This notebook runs the existing, frozen A-OKVQA and OK-VQA protocol at scientific commit `2b9788e09e965ad894a5c29ce81e84cccfa52a7d`. That checkpoint passed 71 CPU software checks earlier; this launcher has only been source/JSON checked and **has not been executed on a GPU**.

Sign-in, GPU allocation and Drive consent are controlled by Google. This notebook does not bypass authorization or session limits. The extended protocol may require multiple sessions. A completed run still requires independent evidence annotation, matched external baselines, scientific review, and author approval before manuscript submission.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, traceback
EXPECTED_COMMIT = '2b9788e09e965ad894a5c29ce81e84cccfa52a7d'
CAL_N = 1000
EVAL_MAX = None
SEED = 2026
SCOPE = 'extended'
MAX_PIXELS = 1003520
DATASETS = ['aokvqa', 'okvqa']
RUN_CONFIG = dict(code_commit=EXPECTED_COMMIT, cal_n=CAL_N, eval_max=EVAL_MAX,
                  seed=SEED, scope=SCOPE, max_pixels=MAX_PIXELS, datasets=DATASETS)
CONFIG_ID = hashlib.sha256(json.dumps(RUN_CONFIG, sort_keys=True).encode()).hexdigest()
RUN_NAME = 'paper4_auto_' + EXPECTED_COMMIT[:8] + '_' + CONFIG_ID[:12]


## 1. Connect the GPU and persistent storage

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a CUDA GPU under Runtime > Change runtime type, then run all again.'
print('GPU:', torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/Paper4Runs') / RUN_NAME
RUN.mkdir(parents=True, exist_ok=True)
config_path = RUN / 'launcher_config.json'
if config_path.exists():
    assert json.loads(config_path.read_text()) == RUN_CONFIG, 'Run configuration changed; existing results were preserved.'
else:
    config_path.write_text(json.dumps(RUN_CONFIG, indent=2))
print('Persistent run directory:', RUN)


## 2. Fetch and freeze the implementation

In [ ]:
REPO = 'https://github.com/junnubabu-ctrl/paper4-selective-kbvqa.git'
ROOT = Path('/content/paper4-science-' + EXPECTED_COMMIT[:12])
try:
    if not (ROOT / '.git').exists():
        if ROOT.exists() and any(ROOT.iterdir()):
            raise RuntimeError('The source directory is nonempty and is not a Git checkout; no files were changed.')
        subprocess.run(['git', 'clone', '--no-checkout', REPO, str(ROOT)], check=True)
        subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', EXPECTED_COMMIT], check=True)
    commit = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
    assert commit == EXPECTED_COMMIT, 'Unexpected scientific code version; existing files were preserved.'
    dirty = subprocess.check_output(['git', '-C', str(ROOT), 'status', '--porcelain', '--untracked-files=no'], text=True).strip()
    assert not dirty, 'Scientific source has changed; existing files were preserved.'
    commit_file = RUN / 'code_commit.txt'
    if commit_file.exists():
        assert commit_file.read_text().strip() == commit, 'This run belongs to a different code version.'
    else:
        commit_file.write_text(commit + '\n')
    os.chdir(ROOT)
    print('Frozen code commit:', commit)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[vlm,retrieval,dev,analysis]'], check=True)
    # Refuse silent core dependency changes when resuming existing predictions.
    from importlib.metadata import version
    core = ['torch', 'transformers', 'accelerate', 'bitsandbytes', 'numpy',
            'scipy', 'scikit-learn', 'sentence-transformers', 'pillow', 'requests']
    env = dict(python=sys.version.split()[0], packages={name: version(name) for name in core})
    lock_path = RUN / 'environment_lock.json'
    if lock_path.exists():
        assert json.loads(lock_path.read_text()) == env, 'Core environment changed. Existing outputs are preserved; inspect versions before resuming.'
    else:
        lock_path.write_text(json.dumps(env, indent=2))
except BaseException:
    (RUN / 'SETUP_ERROR.txt').write_text(traceback.format_exc())
    raise


## 3. Verify software and inspect the experiment plan

In [ ]:
subprocess.run([sys.executable, '-m', 'compileall', '-q', 'scripts', 'src'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
for dataset in DATASETS:
    subprocess.run([sys.executable, 'scripts/run_full_study.py', '--dataset', dataset, '--scope', SCOPE, '--plan'], check=True)


## 4. Run or resume the full protocol
Runs both datasets in order. A failed stage stops execution. Completed predictions and progress records are retained in Drive. At a handled study failure or normal completion, the notebook attempts to save a ZIP bundle in the same run folder. Abrupt VM deletion cannot execute the export, but previously flushed Drive files remain.

Do not infer empirical success from a completed process or the software tests. Review the actual predictions, metrics, data coverage and evidence validity.


In [ ]:
import zipfile
from datetime import datetime, timezone
RUN_STATE = {'status': 'RUNNING', 'code_commit': EXPECTED_COMMIT,
             'started_utc': datetime.now(timezone.utc).isoformat(),
             'manuscript_ready': False}
STATUS_PATH = RUN / 'PAPER4_RUN_STATUS.txt'

def save_status():
    STATUS_PATH.write_text(json.dumps(RUN_STATE, indent=2))

def save_bundle():
    archive = RUN / 'Paper4_Results_Bundle.zip'
    temporary = RUN / 'Paper4_Results_Bundle.zip.partial'
    allowed = {'.json', '.jsonl', '.txt', '.log', '.png', '.pdf', '.yaml', '.yml', '.md'}
    inventory = []
    try:
        with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as z:
            for path in sorted(RUN.rglob('*')):
                relative = path.relative_to(RUN)
                if not path.is_file() or path.is_symlink() or path.suffix not in allowed:
                    continue
                if {'datasets', 'cache', '.git'} & set(relative.parts):
                    continue
                if not path.resolve().is_relative_to(RUN.resolve()):
                    continue
                checksum = hashlib.sha256()
                size = 0
                with path.open('rb') as source, z.open(relative.as_posix(), 'w', force_zip64=True) as target:
                    while chunk := source.read(1024 * 1024):
                        target.write(chunk)
                        checksum.update(chunk)
                        size += len(chunk)
                inventory.append(dict(path=relative.as_posix(), bytes=size, sha256=checksum.hexdigest()))
            z.writestr('BUNDLE_MANIFEST.json', json.dumps(dict(
                files=inventory, manuscript_ready=False,
                exclusions=['datasets', 'retrieval caches', 'model weights'],
                note='Prediction records preserve their evidence; this is not an immutable knowledge snapshot.'
            ), indent=2))
        temporary.replace(archive)
    except BaseException:
        temporary.unlink(missing_ok=True)
        raise
    return archive

save_status()
try:
    for dataset in DATASETS:
        command = [sys.executable, 'scripts/run_full_study.py', '--dataset', dataset,
                   '--out', str(RUN / dataset), '--scope', SCOPE, '--cal-n', str(CAL_N),
                   '--seed', str(SEED), '--max-pixels', str(MAX_PIXELS)]
        if EVAL_MAX is not None:
            command += ['--eval-max', str(EVAL_MAX)]
        subprocess.run(command, check=True)
    for dataset in DATASETS:
        status = json.loads((RUN / dataset / 'completion.json').read_text())
        assert status.get('status') == 'COMPLETED', 'Dataset completion is missing.'
        assert status.get('dataset') == dataset and status.get('scope') == SCOPE, 'Completion metadata mismatch.'
        assert status.get('development') is (EVAL_MAX is not None), 'Development/full-run status mismatch.'
        assert (RUN / dataset / 'report/study_report.json').is_file(), 'Study report is missing.'
        assert (RUN / dataset / 'metrics' / (dataset + '_B0_B5_summary.json')).is_file(), 'Benchmark summary is missing.'
    RUN_STATE['status'] = 'COMPLETED_REQUESTED_PROTOCOL'
except BaseException as error:
    RUN_STATE['status'] = 'INTERRUPTED' if isinstance(error, KeyboardInterrupt) else 'FAILED'
    RUN_STATE['error'] = str(error)
    (RUN / 'EXECUTION_ERROR.txt').write_text(traceback.format_exc())
    raise
finally:
    RUN_STATE['finished_utc'] = datetime.now(timezone.utc).isoformat()
    save_status()
    try:
        print('Saved results bundle:', save_bundle())
    except Exception as export_error:
        (RUN / 'EXPORT_ERROR.txt').write_text(str(export_error))
        print('Bundle export failed; individual result files remain in:', RUN)
    print('Status:', RUN_STATE['status'], '| Manuscript ready: False')


## 5. Download the saved bundle
This downloads the completed bundle. After a handled study error, this cell can be run manually to download the partial bundle. It does not convert partial results into completed experiments.


In [ ]:
from google.colab import files
archive = RUN / 'Paper4_Results_Bundle.zip'
if archive.exists():
    files.download(str(archive))
else:
    print('No bundle yet. Check saved status and logs in:', RUN)
